### Methods

Our methodology is divided into four main stages: Exploratory Data Analysis (EDA), Data Preparation, Baseline Modeling, and an experimental test of a proposed solution.

**1. Exploratory Data Analysis (EDA)**
We began with a comprehensive EDA on the full dataset to understand its characteristics. This analysis revealed several key properties that guided our approach:
*   **Target Imbalance:** The `overall_status` target variable is severely imbalanced, with "Completed" trials significantly outnumbering "Terminated," "Withdrawn," and "Suspended" trials.
*   **Feature Skewness:** Numerical features like `enrollment_count` and the age columns were found to be heavily right-skewed and contained significant outliers.
*   **Data Quality:** We identified quirks in the data, such as the `phases` column containing a meaningful "NA" (Not Applicable) category and the `minimum_age` feature behaving more like a categorical variable than a continuous one.
*   **Embeddings Generation:** We discovered that the pre-computed embeddings were not included in the default dataset and needed to be generated manually. Due to computational constraints, this was performed on a 140,000-row subset using a `thomas-sounack/BioClinical-ModernBERT-base` sentence transformer, and the results were saved for our proof-of-concept models.

**2. Data Preparation and Pipeline**
To ensure a robust and reproducible workflow, we developed a preprocessing pipeline with a strict train/test split to prevent data leakage. All modeling was performed on an 80/20 split of our enhanced 140k-row subset.
*   **Missing Values:** We handled missing data by dropping rows for features with minimal missingness (e.g., `sex`) and imputing with the median for numerical features with moderate missingness (e.g., `minimum_age`).
*   **Feature Encoding:** Categorical features were converted into a numerical format using One-Hot Encoding.
*   **Feature Scaling:** Numerical features were standardized using a `StandardScaler` to have a mean of 0 and a standard deviation of 1.

**3. Baseline Modeling**
We established two distinct baseline models to isolate the predictive power of different data modalities:
*   **Baseline 1 (Structured Data):** A `RandomForestClassifier` was trained on the preprocessed structured metadata. This model served as our primary benchmark and achieved a **weighted F1-score of 0.89**, but showed poor recall (0.48) on the minority "Failure" class.
*   **Baseline 2 (Embeddings):** We first trained a `LogisticRegression` model on the 1536-dimensional concatenated text embeddings, which performed very poorly (Failure F1-score: 0.06). A subsequent `RandomForestClassifier` failed completely, collapsing into a "lazy classifier" that only predicted the majority class (Failure F1-score: 0.00).

**4. Downsampling Experiment**
To test our hypothesis that class imbalance was the primary issue, we conducted a final experiment. We trained a new `RandomForestClassifier` on a **randomly downsampled** version of the embeddings training data. This model showed a dramatic improvement in identifying the minority class, increasing the **recall for "Failure" from 0.00 to 0.63**. This result provides strong evidence that addressing the data distribution is a critical step for our final P3 model.